# max-back-tied-half — faded example 3: Verify the Mass Conservation Invariant for maximum_back

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `max-back-tied-half`. Running the beacon reports progress on the `Backprop: max_back with tied half-mass` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: max_back with tied half-mass` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`max-back-tied-half`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "max-back-tied-half"
DD_SUBTOPIC = "Backprop: max_back with tied half-mass"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The key correctness property of the half-mass convention is that `maximum_back0[i] + maximum_back1[i] == grad_out[i]` at every position `i`. This holds because the masks `(x > y) + 0.5*(x == y)` and `(x < y) + 0.5*(x == y)` sum to exactly 1 everywhere — the winner gets 1, loser gets 0, and ties each get 0.5.

## Faded exercise 3

Complete `verify_mass_conservation` below. The back functions are already defined. Fill in the single blanked line that checks whether `g0 + g1` is close to `grad_out`.

**Fill in:** Check mass conservation using t.allclose on g0 + g1 vs grad_out.

In [ ]:
import torch as t

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def maximum_back1(grad_out, x, y):
    mask = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def verify_mass_conservation(x, y, grad_out):
    """
    Returns True if maximum_back0 + maximum_back1 == grad_out everywhere.
    """
    g0 = maximum_back0(grad_out, x, y)
    g1 = maximum_back1(grad_out, x, y)
    conserves = None  # TODO: Check mass conservation using t.allclose on g0 + g1 vs grad_out.
    return conserves

# Exercise it
t.manual_seed(81)
x = t.randn(6)
y = t.randn(6)
grad_out = t.randn(6)
print(f"Mass conserved: {verify_mass_conservation(x, y, grad_out)}")


import torch as t

def _test():
    # Random case
    t.manual_seed(81)
    x = t.randn(6)
    y = t.randn(6)
    grad_out = t.randn(6)
    assert verify_mass_conservation(x, y, grad_out) is True

    # Case with forced ties
    x2 = t.tensor([1.0, 2.0, 3.0])
    y2 = t.tensor([1.0, 1.0, 4.0])  # tie at pos 0
    g2 = t.ones(3)
    assert verify_mass_conservation(x2, y2, g2) is True

    # Verify the return is bool
    result = verify_mass_conservation(x, y, grad_out)
    assert isinstance(result, bool)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def maximum_back0(grad_out, x, y):
    mask = (x > y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def maximum_back1(grad_out, x, y):
    mask = (x < y).to(grad_out.dtype) + 0.5 * (x == y).to(grad_out.dtype)
    return grad_out * mask

def verify_mass_conservation(x, y, grad_out):
    g0 = maximum_back0(grad_out, x, y)
    g1 = maximum_back1(grad_out, x, y)
    conserves = bool(t.allclose(g0 + g1, grad_out, atol=1e-6))
    return conserves

# Exercise it
t.manual_seed(81)
x = t.randn(6)
y = t.randn(6)
grad_out = t.randn(6)
print(f"Mass conserved: {verify_mass_conservation(x, y, grad_out)}")
```
</details>